# University of Potsdam LLM Proxy connectivity test

This notebook verifies credentials, lists every available model, and makes one minimal inference request.

## 1. Imports

In [ ]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

## 2. Locate the repository and load `.env`

In [ ]:
def find_repo_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in (current, *current.parents):
        if (candidate / ".env").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root containing .env."
    )


repo_root = find_repo_root()
env_path = repo_root / ".env"

load_dotenv(env_path)

api_key = os.getenv("POTSDAM_API_KEY")
base_url = os.getenv("BASE_POTSDAM_LLM_URL")

if not api_key:
    raise RuntimeError("POTSDAM_API_KEY is missing or empty.")

if not base_url:
    raise RuntimeError("BASE_POTSDAM_LLM_URL is missing or empty.")

base_url = base_url.rstrip("/")

print("API key loaded: yes")
print(f"Base URL: {base_url}")

## 3. Query all available models

In [ ]:
from typing import Any


headers = {
    "Authorization": f"Bearer {api_key}",
}


def parse_model_ids(payload: Any) -> list[str]:
    if not isinstance(payload, dict):
        raise ValueError("The model-list response is not a JSON object.")

    entries = payload.get("data")

    if not isinstance(entries, list):
        raise ValueError(
            "The model-list response does not contain a 'data' list."
        )

    model_ids = {
        entry["id"]
        for entry in entries
        if isinstance(entry, dict)
        and isinstance(entry.get("id"), str)
        and entry["id"].strip()
    }

    return sorted(model_ids)


model_endpoint_candidates = [
    ("", f"{base_url}/models"),
    ("/v1", f"{base_url}/v1/models"),
]

available_models = []
api_prefix = None
last_error = None

for prefix, endpoint in model_endpoint_candidates:
    try:
        response = requests.get(
            endpoint,
            headers=headers,
            timeout=60,
        )
    except requests.Timeout as exc:
        raise RuntimeError(
            f"The request to {endpoint} timed out."
        ) from exc
    except requests.ConnectionError as exc:
        raise RuntimeError(
            f"Could not connect to {endpoint}. "
            "Check the base URL, network connection, and UP VPN."
        ) from exc

    if response.status_code in {401, 403}:
        raise RuntimeError(
            f"Authentication or permission failure "
            f"(HTTP {response.status_code})."
        )

    if response.status_code == 429:
        raise RuntimeError(
            "The proxy returned HTTP 429: rate limit or quota exceeded."
        )

    if response.status_code >= 500:
        raise RuntimeError(
            f"The proxy or upstream provider returned "
            f"HTTP {response.status_code}."
        )

    if response.status_code == 404:
        last_error = f"{endpoint} returned HTTP 404."
        continue

    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(
            f"The model-list request failed with "
            f"HTTP {response.status_code}: {response.text[:500]}"
        ) from exc

    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(
            "The model-list endpoint did not return valid JSON."
        ) from exc

    available_models = parse_model_ids(payload)

    if not available_models:
        raise RuntimeError(
            "The request succeeded, but no model IDs were returned."
        )

    api_prefix = prefix
    successful_models_endpoint = endpoint
    break

if not available_models or api_prefix is None:
    raise RuntimeError(
        last_error or "No working model-list endpoint was found."
    )

print(f"Successful endpoint: {successful_models_endpoint}")
print(f"Number of available models: {len(available_models)}")
print("\nAvailable models:")

for index, model_id in enumerate(available_models, start=1):
    print(f"{index}. {model_id}")

## 4. Test one available model

In [ ]:
requested_model = os.getenv("POTSDAM_MODEL")

if requested_model:
    if requested_model not in available_models:
        raise ValueError(
            f"POTSDAM_MODEL='{requested_model}' is unavailable. "
            f"Choose one of the displayed model IDs."
        )

    candidate_models = [requested_model]
else:
    candidate_models = available_models

chat_endpoint = f"{base_url}{api_prefix}/chat/completions"

successful_model = None
assistant_text = None
model_errors = {}

for selected_model in candidate_models:
    payload = {
        "model": selected_model,
        "messages": [
            {
                "role": "user",
                "content": (
                    "Reply with exactly: "
                    "The Potsdam LLM proxy works."
                ),
            }
        ],
    }

    try:
        response = requests.post(
            chat_endpoint,
            headers={
                **headers,
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=120,
        )
    except requests.Timeout:
        model_errors[selected_model] = "request timed out"
        continue
    except requests.ConnectionError:
        model_errors[selected_model] = "connection failed"
        continue

    if response.status_code in {401, 403}:
        raise RuntimeError(
            f"Authentication or permission failure "
            f"(HTTP {response.status_code})."
        )

    if response.status_code == 429:
        raise RuntimeError(
            "The proxy returned HTTP 429: rate limit or quota exceeded."
        )

    if not response.ok:
        model_errors[selected_model] = (
            f"HTTP {response.status_code}: {response.text[:300]}"
        )
        continue

    try:
        response_json = response.json()
        assistant_text = (
            response_json["choices"][0]["message"]["content"]
        )
    except (
        requests.JSONDecodeError,
        KeyError,
        IndexError,
        TypeError,
    ) as exc:
        model_errors[selected_model] = (
            f"unexpected response structure: {exc}"
        )
        continue

    successful_model = selected_model
    break

if successful_model is None:
    raise RuntimeError(
        "The credentials could list models, but no tested model "
        "accepted the chat-completion request.\n"
        f"Model-specific errors: {model_errors}"
    )

print(f"Selected model: {successful_model}")
print(f"Model response: {assistant_text}")